# From config to multi-year video — the CLI and the showcase recipe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/10_cli_multi_year.ipynb)

Notebooks 01–09 call `animate()` from Python. The videos that go into talks — ten years
of Sentinel-2 over a forest, 1080p, 16:9, gap-filled, smoothly interpolated — are not
made that way. Each one is **a YAML file and one command**:

```bash
gee-animation --config config/wne_cinema_ndvi_2018_2026.yaml
```

That form is what you check into git, hand to a colleague, or queue twenty of
overnight. This notebook builds a **two-year cut** the same way, start to finish,
and then shows the full ten-year recipe as commands. Three things make a long run
different from a three-month one, and each gets a section:

- **gaps** — a winter month with no clear scene is borrowed from another year (`pool_years`),
- **pacing** — generated frames between observed months (`interpolate`),
- **fragility** — a ten-year run is one job that can stall; it is rendered in chunks.

Makes live Earth Engine calls under your project. The render in §5 takes a few minutes.

## 1 · Install (Colab only)

In [ ]:
import sys, os, shutil, urllib.request
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package + the gee-animation command
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code

# The CLI is a console script next to this kernel's python; make sure a subprocess finds it.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

# render_chunked.py is a repo script, not part of the package: use the checkout if we
# are inside one, otherwise fetch it from GitHub.
RAW = f"https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/{BRANCH}"
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "render_chunked.py").exists()), None)
if REPO is None:
    urllib.request.urlretrieve(f"{RAW}/scripts/render_chunked.py", "render_chunked.py")
    SCRIPT = Path("render_chunked.py")
else:
    SCRIPT = REPO / "scripts" / "render_chunked.py"

# Joining the chunks needs an ffmpeg binary on PATH; imageio-ffmpeg ships one.
if shutil.which("ffmpeg") is None:
    import imageio_ffmpeg
    bindir = Path(".ffmpeg_bin"); bindir.mkdir(exist_ok=True)
    (bindir / "ffmpeg").unlink(missing_ok=True)
    (bindir / "ffmpeg").symlink_to(imageio_ffmpeg.get_ffmpeg_exe())
    os.environ["PATH"] = str(bindir.resolve()) + os.pathsep + os.environ["PATH"]
print("script:", SCRIPT, "| gee-animation:", shutil.which("gee-animation"), "| ffmpeg:", shutil.which("ffmpeg"))

> **Updated the code on the branch?** After the install cell above, do **Runtime → Restart session**, then **Run all** — otherwise Python keeps the previously-imported (old) module in memory.

## 2 · Authenticate Earth Engine (once)
A separate step — run it a single time. In Colab it opens an `ee.Authenticate()` prompt.
No project of your own yet? [00_setup](00_setup.ipynb#2-·-create-your-own-earth-engine-project)
walks you through creating one (five minutes, free).

The CLI runs as a **separate process** and finds the same stored credentials, so this one
cell covers the subprocess calls below as well.

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · The config, key by key

Everything `animate()` takes as keywords is a key here; the file is the `RunConfig` of
notebook 01 on disk (`config.RunConfig.from_yaml`). The keys that matter for a long run:

| key | what it does | in this cut |
|---|---|---|
| `aoi.frame` / `aoi.region` | the rendered rectangle / the subject polygon that gates clouds and gets outlined | a 22 × 9 km rectangle, the Grumsin polygon |
| `region_max_cloud_percent`, `max_cloud_percent` | the two cloud gates: over the polygon (strict) and per scene (coarse) | 20 % / 70 % |
| `pool_years` + `pool_strategy: gap_fill` | a month with no usable scene borrows the clearest same-month image from another year inside the window | 2022–2026 |
| `render.interpolate` | generated frames between observed months, computed on the index values before colouring | 4 |
| `render.preset` / `render.aspect` | canvas size and shape; the **frame bbox must have the matching shape** or the map is letterboxed | 720p, 16:9 |
| `metadata: true` | per-frame statistics into `out/metadata.db` | on |

The frame rectangle below is the one the showcase configs use: 22.2 × 9.35 km, which is
width/height ≈ 2.37 in metres — the shape a 16:9 canvas has left for the map once the
two-line header and the footer are taken out. For your own site,
`.claude/skills/rendering-showcase-animations/scripts/fit_frame_bbox.py` computes such a
box around a region, with the cos(latitude) correction (degrees of longitude are shorter
than degrees of latitude up here).

In [ ]:
import urllib.request, yaml

urllib.request.urlretrieve(f"{RAW}/docs/aoi/wne/wne.geojson", "wne.geojson")   # the Grumsin polygon

CFG_TEXT = f"""# Two years of Sentinel-2 NDVI over the Grumsin beech forest, gap-filled, interpolated.
name: wne_2yr_cli                       # -> out/wne_2yr_cli.mp4 (+ parts, frames, metadata)
project: {PROJECT}
title: Grumsin Beech Forest
subtitle: UNESCO World Heritage site, Brandenburg, Germany

aoi:
  frame: {{ bbox: [13.7311, 52.9440, 14.0623, 53.0280] }}   # 22.2 x 9.35 km, fills a 16:9 map box
  region: {{ geojson: wne.geojson }}                        # the subject: gates clouds, gets outlined
  region_max_cloud_percent: 20          # keep a scene only if < 20 % cloud over the polygon
start: "2024-01-01"
end: "2026-01-01"
sensor: sentinel2
index: ndvi
cadence: monthly
max_cloud_percent: 70                   # coarse per-scene gate (scene metadata), before the one above
draw_region: true

pool_years: [2022, 2026]                # donors for empty months; widened past the run on both sides
pool_strategy: gap_fill                 # keep every month that has data; borrow only for the gaps

render:
  fps: 2
  interpolate: 4                        # 4 generated frames between observed months (client-side, free)
  scale: 10
  dimensions: 1280                      # thumbnail width; 10 m pixels over 22 km is ~2200, so this downsamples
  crs: auto                             # UTM zone of the AOI: square pixels, honest scale bar
  preset: 720p
  aspect: "16:9"
  quality: 6

metadata: true                          # per-frame stats -> out/metadata.db (plotted in §7)
"""
CFG = Path("wne_2yr_cli.yaml")
CFG.write_text(CFG_TEXT)

from gee_animation.config import RunConfig
cfg = RunConfig.from_yaml(str(CFG))      # the same validation the CLI runs first
print(f"{cfg.name}: {cfg.sensor}/{cfg.index}, {cfg.start}..{cfg.end}, preset {cfg.preset} {cfg.aspect}, "
      f"interpolate {cfg.interpolate}, pool {cfg.pool_years} {cfg.pool_strategy}")

## 4 · Before rendering: what will each month be made of?

`--inventory` does not render. It lists every candidate scene with both cloud numbers
and the reason it was rejected, as `out/<name>_inventory.csv`. It refuses a config with
`pool_years` — the inventory buckets scenes by the nominal calendar, and a pooled run
draws from other years, so the table would contradict the video. The cell writes a
copy of the config without pooling for it. This is also the cheapest way to find out
**why** a month is missing before you spend a render on it.

In [ ]:
inv = yaml.safe_load(CFG_TEXT)
inv.pop("pool_years"); inv.pop("pool_strategy")
inv["name"] = "wne_2yr_cli_inv"
INV = Path("wne_2yr_cli_inv.yaml"); INV.write_text(yaml.safe_dump(inv, sort_keys=False))

!gee-animation --config {INV} --inventory

In [ ]:
import csv
from collections import Counter

rows = list(csv.DictReader(open("out/wne_2yr_cli_inv_inventory.csv")))
print(f"{len(rows)} candidate scenes; rejection reasons:", Counter(r["reason"] for r in rows if r["usable"] != "True"))
by_month = Counter(r["period_label"] for r in rows if r["usable"] == "True")
months = sorted({r["period_label"] for r in rows})
empty = [m for m in months if by_month[m] == 0]
print("usable scenes per month:", dict(sorted(by_month.items())))
print("months with no usable scene (these will be borrowed from another year):", empty or "none")

## 5 · Render it, in chunks

A long run is one fragile job: one `aggregate_array` over every candidate scene, one
process holding the whole result, one network blip at month 80 and nothing is written.
`scripts/render_chunked.py` cuts the date range into pieces, runs `gee-animation` on
each as a child process with its own `name` (`<name>_part01`, `_part02`, …), and joins
the finished MP4s with `ffmpeg -c copy` — no re-encoding. A chunk that fails is retried
alone; every thumbnail already fetched is cached on disk, so a rerun costs minutes.

Two years in yearly chunks is small enough to watch here. The CLI prints only
`wrote <path>` lines, so quiet is normal early; `capping fetch dimensions` in the log
means fetching has started. Expect a few minutes.

In [ ]:
!python -u {SCRIPT} --config {CFG} --years 1

## 6 · What came out

The joined video, the two parts it was made from, and one annotated PNG per observed
month. Look at the status line at the bottom of a winter frame: a borrowed month says
which year its image came from and how many passes went into it — the viewer is never
shown a gap-filled frame as if it were observed.

In [ ]:
from IPython.display import Video, Image, display
import subprocess, shutil

out = Path("out")
parts = sorted(out.glob("wne_2yr_cli_part*.mp4"))
final = out / "wne_2yr_cli.mp4"
print("parts:", [p.name for p in parts], "| joined:", final.name, f"{final.stat().st_size/1e6:.1f} MB")

if shutil.which("ffprobe"):
    n = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=nb_frames",
                        "-of", "csv=p=0", str(final)], capture_output=True, text=True).stdout.strip()
    pngs = sorted(p for p in out.glob("wne_2yr_cli_part*_20??-??.png"))
    obs = len(pngs); per_chunk = [len([p for p in pngs if part.stem in p.stem]) for part in parts]
    print(f"{n} frames in the joined video from {obs} observed months ({per_chunk} per chunk).")
    print(f"Unchunked it would be {obs + (obs - 1) * cfg.interpolate}: interpolation is computed within a chunk,"
          f" so the transition across each boundary is missing — one small jump per seam.")
Video(str(final), embed=True, width=720)

In [ ]:
pngs = sorted(out.glob("wne_2yr_cli_part*_20??-??.png"))
borrowed = [p for p in pngs if p.stem[-7:] in empty]        # `empty` from the inventory in §4
show = borrowed[0] if borrowed else pngs[0]
print(len(pngs), "observed months; showing", show.name, "— a borrowed month, read its status line" if borrowed else "")
display(Image(filename=str(show), width=720))

## 7 · The numbers behind the video

`metadata: true` wrote one row per observed month into `out/metadata.db` (table
`frame_clouds`): scene count, cloud fraction, and the mean / 10th / 90th percentile of
the index inside the polygon and in the frame around it. A chunked run writes its rows
under the part names. This is the inside-versus-outside chart that
[`docs/publishing-animations.md`](../docs/publishing-animations.md) asks for next to
every published animation.

In [ ]:
import sqlite3
import matplotlib.pyplot as plt

con = sqlite3.connect("out/metadata.db")
rows = con.execute("""SELECT month, n_scenes, aoi_mean, outside_mean FROM frame_clouds
                      WHERE name LIKE 'wne_2yr_cli_part%' ORDER BY month""").fetchall()
con.close()
months = [r[0] for r in rows]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(months, [r[2] for r in rows], marker="o", label="inside the polygon")
ax.plot(months, [r[3] for r in rows], marker="s", label="frame outside it")
ax.set_ylabel("mean NDVI"); ax.set_xticks(months[::3]); ax.tick_params(axis="x", rotation=60)
ax.legend(); ax.set_title(f"{cfg.name}: {len(rows)} observed months, scenes per month {min(r[1] for r in rows)}–{max(r[1] for r in rows)}")
plt.tight_layout(); plt.show()

## 8 · The full recipe — ten years, 1080p

This is how the showcase videos in `config/wne_cinema_*_2018_2026.yaml` are made. It is
the same thing as above with three changes: the dates, the preset, and the chunk length.
Not run here — a ten-year Sentinel-2 cut at 1080p is 100 months of fetching plus about
1250 frames to encode, and it should run **alone**: each process already fetches with
several threads, and two renders at once stack onto one Earth Engine quota and draw
HTTP 429 errors.

```bash
# 1. the config: copy the closest one and change the keys that differ
cp config/wne_cinema_ndvi_2018_2026.yaml config/mysite_ndvi_2018_2026.yaml   # edit name, project, aoi, title

# 2. pin which code renders — the console script resolves through the editable
#    install, which may be another checkout; PYTHONPATH reaches the child process too
export PYTHONPATH=$PWD
python -c "import gee_animation; print(gee_animation.__file__)"

# 3. render in five-year chunks: one seam instead of nine (yearly chunking dropped
#    ~180 generated transition frames from a delivered cut and had to be re-rendered)
python -u scripts/render_chunked.py --config config/mysite_ndvi_2018_2026.yaml --years 5

# 4. verify before sharing: a ten-year monthly S2 cut with interpolate: 10 has ~1256
#    frames unchunked; yearly-chunked ones show ~1077 — re-render those
ffprobe -v error -select_streams v:0 -show_entries stream=nb_frames -of csv=p=0 out/mysite_ndvi_2018_2026.mp4
```

What costs a refetch and what is free: thumbnails are cached on disk keyed by everything
that changes what Earth Engine computes. `fps`, `interpolate`, `title`, `subtitle`,
`palette`, `preset`, `aspect`, `quality`, `pixel_grid` and the output switches are **free**
— change them and re-run in minutes. `viz` min/max, `dimensions`, `scale`, `crs`, the
dates, either AOI, the cloud thresholds, `cadence` and pooling **refetch everything**.
So iterate on the look for free, and batch every data-side change into one run.

Three traps the repo learned the hard way:

- **A stalled run looks alive.** After a transient network error a run can sit at 0 % CPU
  for an hour with the process still there. Judge by the newest raw frame in `out/`, not
  by `ps`; no new frame for 20 minutes with the network back means kill the whole tree
  (the script *and* its `gee-animation` child, which keeps spending quota) and relaunch —
  the rerun resumes from cache.
- **`smooth: harmonic` must never be chunked** — each chunk would fit its own seasonal curve.
- **Landsat LST** has unobservable winters; frames freeze in masked holes. `cadence: quarterly`
  pools several passes per frame when the seasonal cycle matters more than monthly detail.

In [ ]:
RUN_FULL = False    # flip to True on a machine you can leave alone for an hour or two
if RUN_FULL:
    full = yaml.safe_load(CFG_TEXT)
    full.update(name="wne_cinema_cli_2018_2026", start="2018-01-01", end="2026-01-01", pool_years=[2018, 2026])
    full["render"].update(preset="1080p", interpolate=10, dimensions=1440, quality=8)
    FULL = Path("wne_cinema_cli_2018_2026.yaml"); FULL.write_text(yaml.safe_dump(full, sort_keys=False))
    !python -u {SCRIPT} --config {FULL} --years 5

## Make it yours

1. **Free re-render.** Change `title`, `fps: 3` and `interpolate: 8` in `CFG_TEXT`, re-run
   §3 and §5, and time it: every thumbnail is a cache hit, so the run is encode time only.
   Then change `region_max_cloud_percent` to 10 and watch the fetch come back.
2. **Disturbance instead of greenness.** Set `index: nbr` and add a `viz: {min: -0.3, max: 0.7}`
   block — the burn ratio drops where canopy is removed (windthrow, logging, fire). The
   registry defaults for some indices (`evi`, `ndwi`, `ndmi`) are a flat −1..1 that wastes
   half the colour ramp; pin `viz` for those too.
3. **Your own site.** Put your polygon as `aoi.region` and build the frame with
   `fit_frame_bbox.py --region W S E N --aspect 2.373`; keep it several kilometres wider
   than the subject. No polygon yet? A bbox works as `region`, but then clouds are gated over
   a rectangle and the outline is a rectangle — get the polygon before publishing.

*The seam behind it:* a config is a `RunConfig`; `animate()` builds the same object from
keywords and the GUI builds it from form fields. Whatever you can set here you can set
there, and a YAML file is the version that survives a hand-over.